# Case RJ - Grupo 2

**Notebook para Realização da Limpeza e Tratamento dos Dados para a Camada Silver da Arquitetura Medalhão**

Nomes dos Integrantes:
- Anna Cristina
- Gabriel Guimarães
- Juan Perri
- Milton Salgado
- Richard
- Yuri

## Sobre este notebook

Lemos cada tabela da bronze, aplicamos a limpeza combinada pelo grupo e gravamos o resultado numa
tabela silver própria — uma seção por tabela: Chamados (sem dono único, centralizado aqui), Clientes
(Yuri), Base de atendentes e Base de motivos (Anna Cristina), Canais (Richard), Custos e Pesquisa de
Satisfação (Milton) e Chamados por Hora (Juan).

A seção 8 (**Consolidado**) junta as 9 tabelas silver gravadas acima numa `base_silver` (uma linha por
`id_chamado`). **Este é o notebook único e definitivo da silver** — não existe mais nenhum outro
notebook de silver (os antigos, um por tabela mais o de junção, foram descartados).

Nomes das tabelas gravadas: `chamados_silver`, `clientes_silver`, `atendentes_silver`, `motivos_silver`,
`canais_silver`, `canais_depara_silver`, `custos_silver`, `pesquisa_satisfacao_silver`,
`chamados_hora_silver` e `base_silver` — todas com `overwrite` (rodar de novo dá o mesmo resultado).

⚠️ **Nenhuma célula exibe linha de `clientes` (CPF/nome/e-mail)** nem nome de atendente — só contagens
e agregados (o `display` da seção 3 descarta `nome_atendente`).

## Configuração

In [0]:
from pyspark.sql import functions as F

CATALOGO = "dados_vcredit"
SCHEMA = "default"

# origem (bronze)
ORIGEM_CHAMADOS = f"{CATALOGO}.{SCHEMA}.chamados_raw"
ORIGEM_CLIENTES = f"{CATALOGO}.{SCHEMA}.clientes_raw"
ORIGEM_ATENDENTES = f"{CATALOGO}.{SCHEMA}.base_atendentes_raw"
ORIGEM_MOTIVOS = f"{CATALOGO}.{SCHEMA}.base_motivos_raw"
ORIGEM_CANAIS = f"{CATALOGO}.{SCHEMA}.canais_raw"
ORIGEM_CHAMADOS_HORA = f"{CATALOGO}.{SCHEMA}.chamados_hora_raw"
ORIGEM_CUSTOS = f"{CATALOGO}.{SCHEMA}.custos_raw"
ORIGEM_PESQUISA = f"{CATALOGO}.{SCHEMA}.pesquisa_satisfacao_raw"

# destino (silver)
DESTINO_CLIENTES = f"{CATALOGO}.{SCHEMA}.clientes_silver"
DESTINO_ATENDENTES = f"{CATALOGO}.{SCHEMA}.atendentes_silver"
DESTINO_MOTIVOS = f"{CATALOGO}.{SCHEMA}.motivos_silver"
DESTINO_CANAIS = f"{CATALOGO}.{SCHEMA}.canais_silver"
DESTINO_DEPARA = f"{CATALOGO}.{SCHEMA}.canais_depara_silver"
DESTINO_CUSTOS = f"{CATALOGO}.{SCHEMA}.custos_silver"
DESTINO_PESQUISA = f"{CATALOGO}.{SCHEMA}.pesquisa_satisfacao_silver"
DESTINO_CHAMADOS_HORA = f"{CATALOGO}.{SCHEMA}.chamados_hora_silver"
DESTINO_CHAMADOS = f"{CATALOGO}.{SCHEMA}.chamados_silver"
DESTINO_BASE = f"{CATALOGO}.{SCHEMA}.base_silver"

## 1. Chamados

Tabela fato — todo mundo usa, ninguém é responsável único por ela. Limpeza:
- `dropDuplicates` por `id_chamado` (2.636 duplicatas idênticas esperadas).
- `canal_chave`/`motivo_chave`: mesma normalização usada nas seções de Canais e de Motivos (sem
  acento/maiúscula), pra já sair pronta pra ligar com `canais_depara_silver`/`motivos_silver`.
- `resolvido` convertido pra booleano (`dq_resolvido_invalido` quando não dá pra interpretar o texto).
- `id_atendente` com placeholder (`0`, `99`, `999`) ou vazio marcado em `dq_atendente_invalido`.
- `status_atendente` **nesta tabela** é o status do `id_atendente` dentro do chamado (`ausente_na_origem`,
  `invalido_na_origem`, `informado_nao_validado_no_cadastro`). Ele só é renomeado para
  `status_id_atendente` na `base_silver` (seção 8), porque `atendentes_silver` tem uma coluna de mesmo
  nome com outro significado (ativo/inativo no cadastro).

In [0]:
from pyspark.sql import functions as F


def gerar_chamados_silver(df_):
    obrigatorias = [
        "id_chamado", "id_cliente", "motivo",
        "canal", "resolvido", "id_atendente"
    ]
    opcionais = [
        "hora_abertura_chamado",
        "hora_inicio_atendimento",
        "hora_finalizacao_atendimento",
        "tempo_espera",
        "tempo_atendimento"
    ]

    nomes = [c.replace("\ufeff", "").strip().lower() for c in df_.columns]

    if len(nomes) != len(set(nomes)):
        raise ValueError("Cabeçalhos duplicados após normalização.")

    df_ = df_.toDF(*nomes)

    ausentes = set(obrigatorias) - set(nomes)
    if ausentes:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(ausentes)}")

    colunas = obrigatorias + [c for c in opcionais if c in nomes]
    entrada = df_.select(
        *[F.col(c).cast("string").alias(c) for c in colunas]
    )

    total_raw = entrada.count()
    if total_raw == 0:
        raise ValueError("Origem vazia; gravação interrompida.")

    # Remove somente cópias exatas, preservando os valores originais.
    d = entrada.dropDuplicates(colunas).select(
        *[F.col(c).alias(f"{c}_bruto") for c in colunas]
    )

    def limpar(c):
        texto = F.trim(
            F.regexp_replace(c.cast("string"), r"[\s\u00a0]+", " ")
        )
        return F.when(
            ~F.lower(texto).isin("", "null", "none", "nan"), texto
        )

    def norm_key(c):
        texto = F.translate(
            F.lower(limpar(c)),
            "áàâãäéèêëíìîïóòôõöúùûüçñ",
            "aaaaaeeeeiiiiooooouuuucn"
        )
        texto = F.regexp_replace(texto, r"[_\-]+", " ")
        return F.trim(F.regexp_replace(texto, r"\s+", " "))

    def chave_canal(c):
        texto = F.translate(
            F.lower(limpar(c)),
            "áàâãäéèêëíìîïóòôõöúùûüç",
            "aaaaaeeeeiiiiooooouuuuc"
        )
        texto = F.regexp_replace(texto, r"\.", "")
        return F.trim(F.regexp_replace(texto, r"[^a-z0-9]+", " "))

    res_key = norm_key(F.col("resolvido_bruto"))
    atendente_original = limpar(F.col("id_atendente_bruto"))

    atendente_invalido = F.coalesce(
        atendente_original.isNull()
        | atendente_original.isin("0", "99", "999")
        | ~atendente_original.rlike(r"^[0-9]+$")
        | atendente_original.rlike(r"^0+$"),
        F.lit(True)
    )

    d = d.select(
        limpar(F.col("id_chamado_bruto")).alias("id_chamado"),
        limpar(F.col("id_cliente_bruto")).alias("id_cliente"),
        chave_canal(F.col("canal_bruto")).alias("canal_chave"),
        norm_key(F.col("motivo_bruto")).alias("motivo_chave"),
        F.when(
            ~atendente_invalido, atendente_original
        ).alias("id_atendente"),
        F.when(
            res_key.isin("sim", "s", "1", "true", "yes", "resolvido"),
            F.lit(True)
        ).when(
            res_key.isin("nao", "n", "0", "false", "nao resolvido"),
            F.lit(False)
        ).otherwise(
            F.lit(None).cast("boolean")
        ).alias("resolvido"),
        *[F.col(c) for c in d.columns]
    )

    d = (
        d
        .withColumn("dq_resolvido_invalido", F.col("resolvido").isNull())
        .withColumn(
            "dq_resolvido_ausente",
            limpar(F.col("resolvido_bruto")).isNull()
        )
        .withColumn("dq_atendente_invalido", atendente_invalido)
        .withColumn(
            "status_resolucao",
            F.when(F.col("resolvido") == True, "resolvido")
             .when(F.col("resolvido") == False, "nao_resolvido")
             .when(F.col("dq_resolvido_ausente"), "nao_informado")
             .otherwise("valor_invalido")
        )
        .withColumn(
            "status_atendente",
            F.when(atendente_original.isNull(), "ausente_na_origem")
             .when(F.col("dq_atendente_invalido"), "invalido_na_origem")
             .otherwise("informado_nao_validado_no_cadastro")
        )
    )

    # Proteção da chave central, sem DataFrame de checks.
    totais = d.agg(
        F.count("*").alias("total"),
        F.countDistinct("id_chamado").alias("ids_distintos"),
        F.sum(
            F.col("id_chamado").isNull().cast("long")
        ).alias("ids_ausentes")
    ).first()

    if totais["ids_ausentes"]:
        raise ValueError("Existem chamados sem ID; gravação interrompida.")

    if totais["total"] != totais["ids_distintos"]:
        raise ValueError(
            "Existem IDs com conteúdos diferentes; revisar antes de publicar."
        )

    print(f"Total Bronze: {total_raw:,}")
    print(f"Total Silver: {totais['total']:,}")
    print(f"Cópias exatas removidas: {total_raw - totais['total']:,}")

    return d


df_ = spark.table(ORIGEM_CHAMADOS)
df_silver = gerar_chamados_silver(df_)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DESTINO_CHAMADOS)
)

display(
    df_silver
    .groupBy("canal_chave", "status_resolucao")
    .count()
    .orderBy("canal_chave", "status_resolucao")
)

Total Bronze: 134,474
Total Silver: 131,838
Cópias exatas removidas: 2,636


canal_chave,status_resolucao,count
atend especializado,nao_informado,31
atend especializado,nao_resolvido,1059
atend especializado,resolvido,1780
atend inicial,nao_informado,112
atend inicial,nao_resolvido,2984
atend inicial,resolvido,3435
atendimento especializado,nao_informado,180
atendimento especializado,nao_resolvido,5063
atendimento especializado,resolvido,8978
atendimento inicial,nao_informado,486


## 2. Clientes

⚠️ **`clientes` tem CPF (o próprio `id_cliente`), nome e e-mail — nome e e-mail não entram em
nenhuma seleção.** `id_cliente` entra só como chave de ligação pra junção final (seção 8) e **nunca é
exibido** em nenhuma célula — a conferência abaixo mostra a tabela sem essa coluna.

Limpeza:
- `id_cliente` vira a chave de dedup (antes a tabela deduplicava pelas outras 4 colunas juntas, sem
  `id_cliente` — isso colapsava clientes diferentes que só coincidiam em idade/região/data).
- `idade` fora de 18-110 (ou vazia) vira inválida.
- `regiao` padronizada (maiúscula, sem espaço nas pontas).
- `data_cadastro` convertida pra `yyyy/MM/dd` (aceita os formatos `dd/MM/yyyy` e `dd-MM-yyyy` de origem).

In [0]:
df_raw = spark.table(ORIGEM_CLIENTES)
print(f"Total de registros na tabela bruta(bronze): {df_raw.count()}")

df_silver = (
    df_raw
    .select(
        F.trim("id_cliente").alias("id_cliente"),
        "data_cadastro",
        "dia_vencimento_fatura",
        "idade",
        "regiao",
    )
    .withColumn("regiao", F.upper(F.trim(F.col("regiao"))))
    .withColumn(
        "idade",
        F.when(
            F.col("idade").isNull() | (F.col("idade") < 18) | (F.col("idade") > 110), F.lit(None)
        )
        .otherwise(F.col("idade"))
    )
    .withColumn(
        "data_cadastro",
        F.date_format(
            F.coalesce(
                F.try_to_date(F.col("data_cadastro"), "dd/MM/yyyy"),
                F.try_to_date(F.col("data_cadastro"), "dd-MM-yyyy")
            ),
            "yyyy/MM/dd"
        )
    )
    .dropDuplicates(["id_cliente"])
)
print(f"Total de registros na tabela tratada(silver): {df_silver.count()}")
print(f"Total de registros duplicados removidos: {df_raw.count() - df_silver.count()}")

(
    df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_CLIENTES)
)

# nunca exibir id_cliente (CPF) em nenhuma celula
display(spark.table(DESTINO_CLIENTES).drop("id_cliente"))

Total de registros na tabela bruta(bronze): 47705
Total de registros na tabela tratada(silver): 47705
Total de registros duplicados removidos: 0


data_cadastro,dia_vencimento_fatura,idade,regiao
2016/07/02,20,57,NORDESTE
2026/04/03,10,62,CENTRO-OESTE
2017/06/07,15,72,CENTRO-OESTE
null,5,31,SUDESTE
2018/11/14,10,40,SUDESTE
2025/03/03,10,54,SUL
2023/09/21,10,48,SUDESTE
2018/10/27,20,52,SUL
2015/04/05,10,86,NORTE
2025/07/03,10,82,SUDESTE


## 3. Base de atendentes

Limpeza:
- `nivel_atendimento` sem padrão (`"N2"`, `"Nível 1"`, `"1"`...) — extrai o dígito e aceita só `1`/`2`.
- `data_admissao`/`data_desligamento` convertidas de texto (`dd/MM/yyyy`) pra `date`; vazia vira `null`.

In [0]:
df_raw = spark.table(ORIGEM_ATENDENTES)
total_raw = df_raw.count()
print(f"Total de registros na tabela bruta(bronze): {total_raw}")

nivel_digito = F.regexp_extract(F.trim(F.col("nivel_atendimento").cast("string")), r"(\d)", 1)

df_silver = (
    df_raw
    .select(
        "id_atendente",
        "nome_atendente",
        "nivel_atendimento",
        "data_admissao",
        "data_desligamento",
        "status_atendente",
    )
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
    .withColumn(
        "nivel_atendimento",
        F.when(nivel_digito.isin("1", "2"), nivel_digito.cast("int"))
        .otherwise(F.lit(None))
    )
    .withColumn(
        "data_admissao",
        F.try_to_date(F.trim(F.col("data_admissao")), "dd/MM/yyyy")
    )
    .withColumn(
        "data_desligamento",
        F.try_to_date(F.trim(F.col("data_desligamento")), "dd/MM/yyyy")
    )
    .distinct()
)

total_silver = df_silver.count()
print(f"Total de registros na tabela tratada(silver): {total_silver}")
print(f"Total de registros duplicados removidos: {total_raw - total_silver}")

(
    df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_ATENDENTES)
)

# nome_atendente e dado pessoal: nao exibir
display(spark.table(DESTINO_ATENDENTES).drop("nome_atendente"))

Total de registros na tabela bruta(bronze): 25
Total de registros na tabela tratada(silver): 25
Total de registros duplicados removidos: 0


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente
23,Solange Caldeira,1,2025-11-24,null,ativo
8,Josefa Brito,1,2023-01-25,null,ativo
22,Renato Prado,2,2025-07-15,null,ativo
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo
12,Marilene Nascimento,1,2021-11-25,null,ativo
13,Daniel Fonseca,1,2019-06-19,null,ativo
14,Patrícia Barbosa,2,2019-08-24,null,ativo
4,Solange Silveira,1,2019-10-07,null,ativo
5,Joaquim Barros,1,2020-12-06,2026-05-23,inativo
6,Roberta Assunção,1,2021-10-29,null,ativo


## 4. Base de motivos

Limpeza:
- `criticidade` padronizada em minúsculo (misturava `"Alta"`/`"alta"`).
- `categoria` vazia vira `"Outro"` (nunca fica sem categoria).
- `nome_motivo` mantido como veio, mesmo o mais extenso — é descrição válida, não erro.

In [0]:
df_raw = spark.table(ORIGEM_MOTIVOS)
total_raw = df_raw.count()
print(f"Total de registros na tabela bruta(bronze): {total_raw}")

df_silver = (
    df_raw
    .select(
        "id_motivo",
        "nome_motivo",
        "categoria",
        "criticidade",
    )
    .withColumn("nome_motivo", F.trim(F.col("nome_motivo")))
    .withColumn("criticidade", F.lower(F.trim(F.col("criticidade"))))
    .withColumn(
        "categoria",
        F.when(
            F.col("categoria").isNull() | (F.trim(F.col("categoria")) == ""), F.lit("Outro")
        )
        .otherwise(F.trim(F.col("categoria")))
    )
    .distinct()
)

total_silver = df_silver.count()
print(f"Total de registros na tabela tratada(silver): {total_silver}")
print(f"Total de registros duplicados removidos: {total_raw - total_silver}")

(
    df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_MOTIVOS)
)

display(spark.table(DESTINO_MOTIVOS))

Total de registros na tabela bruta(bronze): 14
Total de registros na tabela tratada(silver): 14
Total de registros duplicados removidos: 0


id_motivo,nome_motivo,categoria,criticidade
9,Problema com aplicativo,Atendimento,media
10,Consulta de fatura,Financeiro,baixa
5,Consulta de limite,Financeiro,baixa
12,Contratação de cartão adicional,Outro,media
7,Compra não autorizada,Cartao,alta
3,Contestação de fatura,Financeiro,alta
11,Bloqueio de cartão,Cartao,alta
14,Transferência de agência,Outro,media
16,Cancelamento de conta,Outro,alta
6,Desbloqueio de cartão,Cartao,alta


## 5. Canais

Limpeza:
- Nome do canal normalizado pro padrão oficial, via `ALIASES` (cobre as variações de maiúscula,
  abreviação e apelido que aparecem em `chamados.canal` — se a conferência abaixo achar uma variação
  sem correspondência, é só incluir aqui).
- Duplicata por `nome_canal` removida (fica a de menor `id_canal`).

In [0]:
from pyspark.sql import Window


def chave_canal(coluna):
    c = F.lower(F.trim(coluna))
    c = F.translate(c, "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc")
    c = F.regexp_replace(c, r"\.", "")
    c = F.regexp_replace(c, r"[^a-z0-9]+", " ")
    return F.trim(c)


# variações encontradas em chamados.canal (já na forma normalizada) -> nome padrão do canal.
ALIASES = {
    "ura": "URA",
    "chatbot": "CHATBOT",
    "chat bot": "CHATBOT",
    "atendimento inicial": "ATENDIMENTO INICIAL",
    "atend inicial": "ATENDIMENTO INICIAL",
    "atendimento especializado": "ATENDIMENTO ESPECIALIZADO",
    "atend especializado": "ATENDIMENTO ESPECIALIZADO",
}
MAPA = F.create_map(*[F.lit(x) for par in ALIASES.items() for x in par])


def nome_padrao(coluna):
    return F.coalesce(MAPA[chave_canal(coluna)], F.upper(F.trim(coluna)))


df_raw = spark.table(ORIGEM_CANAIS)
n_raw = df_raw.count()

w = Window.partitionBy("nome_canal").orderBy(F.col("id_canal").asc())

df_silver = (
    df_raw
    .withColumn("id_canal", F.col("id_canal").cast("int"))
    .withColumn("nome_canal", nome_padrao(F.col("nome_canal")))
    .withColumn(
        "tipo_canal",
        F.when(chave_canal(F.col("tipo_canal")) == "autosservico", F.lit("Autosserviço"))
         .otherwise(F.initcap(F.trim(F.col("tipo_canal"))))
    )
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
    .withColumn("is_ativo", F.col("status_canal") == "ativo")
    .filter(F.col("id_canal").isNotNull() & (F.col("nome_canal") != ""))
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .select(
        "id_canal", "nome_canal", "tipo_canal", "status_canal", "is_ativo",
        F.current_timestamp().alias("_treatment_ts"),
    )
)

n_silver = df_silver.count()
print(f"Total de registros na tabela bruta (bronze): {n_raw}")
print(f"Total de registros na tabela tratada (silver): {n_silver}")
print(f"Registros removidos (duplicados/inválidos): {n_raw - n_silver}")

(
    df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_CANAIS)
)

display(spark.table(DESTINO_CANAIS).orderBy("id_canal"))

Total de registros na tabela bruta (bronze): 4
Total de registros na tabela tratada (silver): 4
Registros removidos (duplicados/inválidos): 0


id_canal,nome_canal,tipo_canal,status_canal,is_ativo,_treatment_ts
1,URA,Autosserviço,ativo,true,2026-10-21T02:02:43.366Z
2,CHATBOT,Autosserviço,ativo,true,2026-10-21T02:02:43.366Z
3,ATENDIMENTO INICIAL,Humano,ativo,true,2026-10-21T02:02:43.366Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,true,2026-10-21T02:02:43.366Z


### De-para de canal (variação em `chamados` → `id_canal`)

Guarda cada chave de `ALIASES` já ligada ao `id_canal` correspondente — na junção final, basta
normalizar `chamados.canal` com a mesma `chave_canal()` e fazer o join aqui pra trocar texto por
`id_canal`.

In [0]:
df_depara = (
    spark.createDataFrame(list(ALIASES.items()), ["canal_chave", "nome_canal"])
    .join(spark.table(DESTINO_CANAIS).select("id_canal", "nome_canal"), "nome_canal", "left")
    .select("canal_chave", "id_canal", "nome_canal")
)

assert df_depara.filter(F.col("id_canal").isNull()).count() == 0, "alias aponta para canal inexistente na silver"

(
    df_depara.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_DEPARA)
)

display(spark.table(DESTINO_DEPARA).orderBy("id_canal", "canal_chave"))

canal_chave,id_canal,nome_canal
ura,1,URA
chat bot,2,CHATBOT
chatbot,2,CHATBOT
atend inicial,3,ATENDIMENTO INICIAL
atendimento inicial,3,ATENDIMENTO INICIAL
atend especializado,4,ATENDIMENTO ESPECIALIZADO
atendimento especializado,4,ATENDIMENTO ESPECIALIZADO


## 6. Custos e Pesquisa de Satisfação

**Custos**: `custo` vem em formatos mistos (vírgula/ponto, símbolo de moeda, sufixo "reais"). Vírgula
é sempre separador decimal; ponto só é separador de milhar quando a vírgula também aparece — nunca
remover ponto indiscriminadamente, senão o valor é inflado. Guardado como `decimal(12,2)` (não
`double`), pra não ter ruído de ponto flutuante na soma. `id_custo` duplicado é removido antes de
gravar.

**Pesquisa de satisfação**: nota só é válida de 1 a 5 — fora disso (incluindo vazio) vira inválida,
marcada em `dq_nota_invalida`, nunca descartada. Além da nota estrita, `nota_ajustada` trata `6` como
`5` e `-1`/`0` como `1` (erro de clique acima/abaixo da escala); vazio ou `"NULL"` literal continua
`NULL` nas duas colunas — é "não teve avaliação", não nota ruim. As duas colunas ficam na tabela, como
premissa explícita, sem substituir a regra que o resto do grupo já usa.

In [0]:
def _limpa_nomes(df):
    for antigo in df.columns:
        novo = antigo.replace("\ufeff", "").strip().lower()
        if novo != antigo:
            df = df.withColumnRenamed(antigo, novo)
    return df


df_custos_raw = _limpa_nomes(spark.table(ORIGEM_CUSTOS))

_bruto_custo = F.regexp_replace(F.trim(F.col("custo")), r"[^0-9,.\-]", "")
_numero_custo = F.when(
    _bruto_custo.contains(","),
    F.regexp_replace(F.regexp_replace(_bruto_custo, r"\.", ""), ",", "."),
).otherwise(_bruto_custo)

custos_silver = (
    df_custos_raw
    .withColumn("custo", _numero_custo.try_cast("decimal(12,2)"))
    .select(
        F.trim("id_custo").alias("id_custo"),
        F.trim("id_chamado").alias("id_chamado"),
        "custo",
    )
    .withColumn("dq_custo_invalido", F.col("custo").isNull())
    .withColumn("dq_custo_nao_positivo", F.col("custo") <= 0)
    .dropDuplicates(["id_custo"])
)

(
    custos_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_CUSTOS)
)

print(f"custos: {df_custos_raw.count():,} -> {spark.table(DESTINO_CUSTOS).count():,} linhas (esperado: 130.288)")
print(f"custo invalido (flag): {spark.table(DESTINO_CUSTOS).where('dq_custo_invalido').count():,}")

custos: 131,069 -> 130,288 linhas (esperado: 130.288)
custo invalido (flag): 0


In [0]:
df_pesquisa_raw = _limpa_nomes(spark.table(ORIGEM_PESQUISA))

pesquisa_tipada = df_pesquisa_raw.withColumn("nota_i", F.trim(F.col("nota_atendimento")).try_cast("int"))

_nota_ajustada = (
    F.when(F.col("nota_i").isNull(), F.lit(None))
     .when(F.col("nota_i") == 6, F.lit(5))
     .when(F.col("nota_i").isin(-1, 0), F.lit(1))
     .when(F.col("nota_i").between(1, 5), F.col("nota_i"))
     .otherwise(F.lit(None))
)

pesquisa_bruta = (
    pesquisa_tipada
    .select(
        F.trim("id_pesquisa").alias("id_pesquisa"),
        F.trim("id_chamado").alias("id_chamado"),
        F.when(F.col("nota_i").between(1, 5), F.col("nota_i")).alias("nota"),
        _nota_ajustada.alias("nota_ajustada"),
    )
    .withColumn("dq_nota_invalida", F.col("nota").isNull())
)

_janela = Window.partitionBy("id_chamado").orderBy(F.col("nota").isNull().asc(), F.col("id_pesquisa").asc())
pesquisa_silver = (
    pesquisa_bruta.withColumn("_rank", F.row_number().over(_janela))
    .where(F.col("_rank") == 1).drop("_rank")
)

(
    pesquisa_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_PESQUISA)
)

print(f"pesquisa_satisfacao: {df_pesquisa_raw.count():,} -> {spark.table(DESTINO_PESQUISA).count():,} linhas (1 por id_chamado)")
print(f"nota invalida (flag): {spark.table(DESTINO_PESQUISA).where('dq_nota_invalida').count():,}")
print(f"nota_ajustada preenchida: {spark.table(DESTINO_PESQUISA).where('nota_ajustada is not null').count():,}")

pesquisa_satisfacao: 81,021 -> 80,219 linhas (1 por id_chamado)
nota invalida (flag): 6,836
nota_ajustada preenchida: 74,592


## 7. Chamados por Hora

Fonte da verdade dos tempos (nunca `chamados.tempo_atendimento`, truncado). Mantém os textos brutos,
faz o parsing e aplica duas correções controladas:

- **Ano da abertura**: quando a abertura é depois da finalização e os anos diferem, testa o ano da
  finalização (ou o anterior, se a abertura foi em dezembro e a finalização em janeiro) — só aceita se
  a cadeia de timestamps resultante fica válida. Cadeias já corretas não são alteradas.
- **Virada de dia**: hipótese de que abertura/início às 22h-23h com finalização às 00h-01h é o mesmo
  atendimento cruzando a meia-noite sem registrar a troca de dia — soma 1 dia na finalização só quando
  as três marcações batem na mesma data original, a duração resultante é positiva e cabe no limite
  (`HORA_LIMITE_VIRADA_HORAS`, padrão 4h). Pode ser desligada em `HORA_APLICAR_CORRECAO_DIA`.

Bloqueia entrada vazia ou com `id_chamado` ausente/duplicado. Não inclui a bateria completa de
comparação de execuções do notebook original — só o resultado final e um resumo agregado (sem dado de
cliente). Precisa rodar antes da seção 8 (Consolidado), que lê esta tabela.

### Configuração

In [0]:
# Mantém a hipótese de correção de madrugada do notebook original.
HORA_APLICAR_CORRECAO_DIA = True
HORA_LIMITE_VIRADA_HORAS = 4
HORA_MAPA = {
    "Hora_Abertura_Chamado": "abertura",
    "Hora_Inicio_Atendimento": "inicio",
    "Hora_Finalizacao_Atendimento": "fim",
}
HORA_VERSAO_BRONZE = int(spark.sql(
    f"DESCRIBE HISTORY {ORIGEM_CHAMADOS_HORA} LIMIT 1"
).first()["version"])

### Funções de limpeza e correção

In [0]:
def hora_limpar(df):
    d = df.select(F.trim(F.col("ID_Chamado").cast("string")).alias("id_chamado"),
                  F.trim(F.col("ID_Cliente").cast("string")).alias("id_cliente"),
                  *[F.col(c).cast("string").alias(f"{p}_bruto") for c, p in HORA_MAPA.items()])
    for p in HORA_MAPA.values():
        txt = F.trim(F.regexp_replace(F.col(f"{p}_bruto"), r"[\s\u00a0]+", " "))
        txt = F.regexp_replace(txt, r"(?iu)(?<=\d)\s*[àa]s\s*(?=\d)", " ")
        d = d.withColumn(f"{p}_txt", F.when(F.length(txt) > 0, txt))
    return d


def hora_tipar(df):
    d = df
    padrao = r"^[0-9]{2}/[0-9]{2}/[0-9]{4} (?:[01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$"
    for p in HORA_MAPA.values():
        d = d.withColumn(f"{p}_ts", F.when(F.col(f"{p}_txt").rlike(padrao),
            F.try_to_timestamp(F.col(f"{p}_txt"), F.lit("dd/MM/yyyy HH:mm:ss"))))
        d = d.withColumn(f"{p}_erro_parse",
            F.col(f"{p}_txt").isNotNull() & F.col(f"{p}_ts").isNull())
    return d.withColumn("abandonado", F.col("inicio_txt").isNull())


def hora_corrigir_ano(df):
    d = df.withColumn("abertura_original_ts", F.col("abertura_ts"))
    d = d.withColumn("correcao_ano_candidata", F.coalesce(
        (F.col("abertura_ts") > F.col("fim_ts")) &
        (F.year("abertura_ts") != F.year("fim_ts")), F.lit(False)))
    # Dezembro pode pertencer ao ano anterior à finalização em janeiro.
    # Não altera uma cadeia temporal que já esteja correta.
    ano_candidato = F.when(
        (F.month("abertura_ts") == 12) & (F.month("fim_ts") == 1),
        F.year("fim_ts") - 1,
    ).otherwise(F.year("fim_ts"))
    texto = F.concat(ano_candidato.cast("string"),
                     F.date_format("abertura_ts", "-MM-dd HH:mm:ss"))
    d = d.withColumn("abertura_candidata_ts", F.try_to_timestamp(texto, F.lit("yyyy-MM-dd HH:mm:ss")))
    aceitavel = (F.col("abertura_candidata_ts").isNotNull() &
        (F.col("abertura_candidata_ts") <= F.col("fim_ts")) &
        (F.col("abandonado") | ((F.col("abertura_candidata_ts") <= F.col("inicio_ts")) &
                                    (F.col("inicio_ts") <= F.col("fim_ts")))))
    d = d.withColumn("ano_abertura_corrigido", F.coalesce(
        F.col("correcao_ano_candidata") & aceitavel, F.lit(False)))
    d = d.withColumn("ano_abertura_corrigido_virada_ano",
        F.col("ano_abertura_corrigido") &
        (F.month("abertura_original_ts") == 12) & (F.month("fim_ts") == 1))
    return d.withColumn("abertura_ts", F.when(F.col("ano_abertura_corrigido"),
        F.col("abertura_candidata_ts")).otherwise(F.col("abertura_original_ts")))


def hora_tratar_virada_dia(df, aplicar=True, limite_horas=4):
    """Virada próxima da meia-noite: abertura/início 22-23h, fim 00-01h.

    Soma um dia somente se os três timestamps tiverem a mesma data original,
    início >= abertura, fim < abertura e atendimento proposto positivo e <= limite.
    Preserva o original e é idempotente: uma segunda chamada não soma outro dia.
    """
    if not isinstance(aplicar, bool):
        raise ValueError("aplicar deve ser True ou False")
    if not isinstance(limite_horas, (int, float)) or not 0 < limite_horas <= 4:
        raise ValueError("limite_horas deve ser maior que zero e no máximo 4 nesta regra")
    d = df if "fim_original_ts" in df.columns else df.withColumn("fim_original_ts", F.col("fim_ts"))
    # Aritmética nativa de timestamp: conserva data, hora e frações sem remontar strings.
    d = d.withColumn("fim_candidato_dia_ts", F.expr("fim_original_ts + INTERVAL 1 DAY"))
    d = d.withColumn("duracao_candidata_seg", F.expr("timestampdiff(SECOND, inicio_ts, fim_candidato_dia_ts)"))
    candidato = (
        F.col("abertura_ts").isNotNull() & F.col("inicio_ts").isNotNull() & F.col("fim_original_ts").isNotNull() &
        F.hour("abertura_ts").isin(22, 23) & F.hour("inicio_ts").isin(22, 23) &
        F.hour("fim_original_ts").isin(0, 1) &
        (F.col("inicio_ts") >= F.col("abertura_ts")) &
        (F.to_date("abertura_ts") == F.to_date("inicio_ts")) &
        (F.to_date("inicio_ts") == F.to_date("fim_original_ts")) &
        (F.col("fim_original_ts") < F.col("abertura_ts")) &
        (F.col("duracao_candidata_seg") > 0) &
        (F.col("duracao_candidata_seg") <= F.lit(limite_horas * 3600)))
    d = d.withColumn("virada_dia_candidata", F.coalesce(candidato, F.lit(False)))
    d = d.withColumn("limite_virada_horas", F.lit(float(limite_horas)))
    d = d.withColumn("fim_ts", F.when(F.lit(aplicar) & F.col("virada_dia_candidata"),
                                     F.col("fim_candidato_dia_ts")).otherwise(F.col("fim_ts")))
    d = d.withColumn("dia_fim_corrigido", F.col("virada_dia_candidata") &
                     F.col("fim_ts").eqNullSafe(F.col("fim_candidato_dia_ts")))
    d = d.withColumn("status_virada_dia",
        F.when(F.col("dia_fim_corrigido"), "aplicada_regra_22_23_para_00_01")
         .when(F.col("virada_dia_candidata"), "proposta_nao_aplicada")
         .when(F.col("fim_original_ts") < F.col("inicio_ts"), "inversao_fora_da_regra")
         .otherwise("sem_proposta"))
    # 24:12 e 25:00 são representações relativas ao dia de abertura, não timestamps.
    dias = F.datediff(F.to_date("fim_ts"), F.to_date("abertura_ts"))
    valido = ((F.col("abertura_ts") <= F.col("inicio_ts")) & (F.col("inicio_ts") <= F.col("fim_ts")))
    horas = F.hour("fim_ts") + dias * 24
    return d.withColumn("fim_hora_estendida", F.when(valido,
        F.concat(F.format_string("%02d", horas), F.date_format("fim_ts", ":mm:ss"))))


def hora_calcular(df):
    d = df.withColumn("espera_valida", F.coalesce(F.col("inicio_ts") >= F.col("abertura_ts"), F.lit(False)))
    d = d.withColumn("atendimento_valido", F.coalesce(F.col("espera_valida") &
        (F.col("fim_ts") >= F.col("inicio_ts")), F.lit(False)))
    d = d.withColumn("total_valido", F.coalesce(F.col("fim_ts") >= F.col("abertura_ts"), F.lit(False)))
    # Quando a condição não é válida, when deixa a duração como NULL.
    d = d.withColumn("tempo_espera_seg", F.when(F.col("espera_valida"),
        F.expr("timestampdiff(SECOND, abertura_ts, inicio_ts)")))
    d = d.withColumn("tempo_atendimento_seg", F.when(F.col("atendimento_valido"),
        F.expr("timestampdiff(SECOND, inicio_ts, fim_ts)")))
    d = d.withColumn("tempo_total_seg", F.when(F.col("total_valido"),
        F.expr("timestampdiff(SECOND, abertura_ts, fim_ts)")))
    d = (d.withColumn("data_abertura", F.to_date("abertura_ts"))
          .withColumn("ano_abertura", F.year("abertura_ts"))
          .withColumn("mes_abertura", F.month("abertura_ts"))
          .withColumn("dia_abertura", F.dayofmonth("abertura_ts"))
          .withColumn("hora_abertura", F.hour("abertura_ts"))
          .withColumn("fonte_tempos", F.lit(ORIGEM_CHAMADOS_HORA))
          .withColumn("versao_bronze", F.lit(HORA_VERSAO_BRONZE).cast("long")))
    for p in HORA_MAPA.values():
        d = d.withColumn(f"{p}_formatada", F.date_format(f"{p}_ts", "dd/MM/yyyy HH:mm:ss"))
    d = d.withColumn("data_abertura_formatada", F.date_format("data_abertura", "dd/MM/yyyy"))
    return d.withColumn("faixa_horaria",
        F.when(F.col("hora_abertura").isNull(), F.lit(None).cast("string"))
         .when(F.col("hora_abertura") < 6, "Madrugada")
         .when(F.col("hora_abertura") < 12, "Manhã")
         .when(F.col("hora_abertura") < 18, "Tarde")
         .otherwise("Noite"))

### Execução, validação e gravação

In [0]:
# Lê uma versão fixa da Bronze; não altera a origem.
hora_raw = (
    spark.read.option("versionAsOf", HORA_VERSAO_BRONZE)
    .table(ORIGEM_CHAMADOS_HORA)
    .select("ID_Chamado", "ID_Cliente", *HORA_MAPA.keys())
)
hora_limpo = hora_limpar(hora_raw)

hora_entrada = hora_limpo.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_chamado").alias("ids_distintos"),
    F.coalesce(F.sum(F.when(
        F.col("id_chamado").isNull() | (F.col("id_chamado") == ""), 1
    ).otherwise(0)), F.lit(0)).alias("ids_invalidos"),
).first()
if hora_entrada["total"] == 0:
    raise ValueError("Chamados por hora: origem vazia.")
if hora_entrada["ids_invalidos"] or hora_entrada["ids_distintos"] != hora_entrada["total"]:
    raise ValueError("Chamados por hora: ID ausente ou duplicado após normalização.")

hora_base = hora_corrigir_ano(hora_tipar(hora_limpo))
hora_silver = hora_calcular(hora_tratar_virada_dia(
    hora_base,
    aplicar=HORA_APLICAR_CORRECAO_DIA,
    limite_horas=HORA_LIMITE_VIRADA_HORAS,
))

# As flags preservam problemas da fonte; durações inválidas ficam nulas.
# Nenhum tempo é obtido de chamados.tempo_atendimento.
(
    hora_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DESTINO_CHAMADOS_HORA)
)

### Conferência

In [0]:
# Exibe somente agregados, sem identificadores de clientes.
hora_gravada = spark.table(DESTINO_CHAMADOS_HORA)
display(hora_gravada.agg(
    F.count("*").alias("total_chamados"),
    F.sum(F.col("abandonado").cast("long")).alias("abandonados"),
    F.sum(F.col("ano_abertura_corrigido").cast("long")).alias("anos_corrigidos"),
    F.sum(F.col("dia_fim_corrigido").cast("long")).alias("viradas_aplicadas"),
    F.sum(F.col("ano_abertura_corrigido_virada_ano").cast("long")).alias("anos_corrigidos_dez_jan"),
    F.sum(F.col("abertura_erro_parse").cast("long")).alias("erros_parse_abertura"),
    F.sum(F.col("inicio_erro_parse").cast("long")).alias("erros_parse_inicio"),
    F.sum(F.col("fim_erro_parse").cast("long")).alias("erros_parse_fim"),
    F.count("tempo_espera_seg").alias("n_espera_valida"),
    F.count("tempo_atendimento_seg").alias("n_atendimento_valido"),
    (F.avg("tempo_espera_seg") / 60).alias("tme_min"),
    (F.avg("tempo_atendimento_seg") / 60).alias("tma_min"),
))

total_chamados,abandonados,anos_corrigidos,viradas_aplicadas,anos_corrigidos_dez_jan,erros_parse_abertura,erros_parse_inicio,erros_parse_fim,n_espera_valida,n_atendimento_valido,tme_min,tma_min
131838,3414,198,1,0,0,0,0,128424,128424,3.60069301688158,11.031754708361884


## 8. Consolidado

Junta as 9 tabelas silver que este notebook já gravou numa `base_silver`, uma linha por `id_chamado`
— sem limpar nada de novo, só lendo e ligando o que já foi escrito acima.

- **Canal**: `chamados_silver.canal_chave` → `canais_depara_silver` → `id_canal` → `canais_silver`.
- **Motivo**: `motivos_silver.nome_motivo` normalizado (mesma `norm_key` da seção 1) vira
  `motivo_chave`, ligado direto com `chamados_silver.motivo_chave` — testado contra os dados reais:
  100% de correspondência, nenhum motivo sem match.
- **Atendente**: join direto por `id_atendente`. `chamados_silver.status_atendente` (status do
  `id_atendente` dentro do chamado: ausente/inválido/não validado) e
  `atendentes_silver.status_atendente` (ativo/inativo no cadastro) são coisas diferentes com o
  mesmo nome — a de `chamados_silver` vira `status_id_atendente` aqui pra não colidir.
- **Cliente**: join por `id_cliente` (o próprio CPF) — usado só pra ligar a linha; a coluna é
  descartada logo depois do join, nunca fica na tabela final. Ao contrário de canal/motivo, aqui a
  correspondência **não é 100%**: existem `id_cliente` em `chamados` que não aparecem em `clientes`
  — é um achado de qualidade de dado real, marcado em `dq_cliente_sem_correspondencia` (mesmo padrão
  das outras flags `dq_*` do projeto: nunca descartar a linha, só sinalizar).
- **Custo**: soma de `custos_silver.custo` por `id_chamado`, considerando só lançamentos **> 0**
  (exclui nulo/inválido e também `<= 0` — mesma regra combinada pra gold).
- **Nota**: `nota`/`nota_ajustada` de `pesquisa_satisfacao_silver`. `nota` já é nula quando
  `dq_nota_invalida`; `nota_ajustada` é mantida mesmo nesses casos — é justamente ela que recupera
  `6→5` e `0`/`-1→1`.
- **Tempos**: `tempo_espera_seg`/`tempo_atendimento_seg`/`abandonado`/`faixa_horaria` etc. de
  `chamados_hora_silver`.

⚠️ Nenhuma célula mostra linha de `clientes` — só contagens e agregados.

In [0]:
def norm_key(c):
    # mesma normalizacao da secao 1 (Chamados) - precisa bater com motivo_chave de la.
    texto = F.translate(
        F.lower(F.trim(c.cast("string"))),
        "áàâãäéèêëíìîïóòôõöúùûüçñ",
        "aaaaaeeeeiiiiooooouuuucn"
    )
    texto = F.regexp_replace(texto, r"[_\-]+", " ")
    return F.trim(F.regexp_replace(texto, r"\s+", " "))


chamados_s = spark.table(DESTINO_CHAMADOS).select(
    "id_chamado", "id_cliente", "canal_chave", "motivo_chave", "id_atendente", "resolvido",
    "dq_resolvido_invalido", "dq_atendente_invalido", "status_resolucao",
    # renomeada - colide com base_atendentes_silver.status_atendente (significado diferente)
    F.col("status_atendente").alias("status_id_atendente"),
)

# marca quem tem cliente correspondente antes do join, sem depender de nenhuma coluna de negocio
# especifica (regiao/idade podem legitimamente vir nulas na propria clientes_silver).
clientes_s = spark.table(DESTINO_CLIENTES).withColumn("_cliente_encontrado", F.lit(True))
atendentes_s = spark.table(DESTINO_ATENDENTES).withColumnRenamed("nivel_atendimento", "nivel_atendente")
motivos_s = spark.table(DESTINO_MOTIVOS).withColumn("motivo_chave", norm_key(F.col("nome_motivo")))
canais_s = spark.table(DESTINO_CANAIS).drop("_treatment_ts")
canais_de_para_s = spark.table(DESTINO_DEPARA).select("canal_chave", "id_canal")
custos_s = spark.table(DESTINO_CUSTOS)
pesquisa_s = spark.table(DESTINO_PESQUISA)
hora_s = spark.table(DESTINO_CHAMADOS_HORA).select(
    "id_chamado", "tempo_espera_seg", "tempo_atendimento_seg", "tempo_total_seg",
    "abandonado", "data_abertura", "mes_abertura", "hora_abertura", "faixa_horaria",
    "ano_abertura_corrigido", "dia_fim_corrigido",
)

custo_por_chamado = (
    custos_s.where(F.col("custo") > 0)  # exclui nulo/invalido E <= 0 - mesma regra combinada pra gold
    .groupBy("id_chamado").agg(F.sum("custo").alias("custo"))
)

base_silver = (
    chamados_s
    .join(canais_de_para_s, "canal_chave", "left").drop("canal_chave")
    .join(canais_s, "id_canal", "left")
    .join(motivos_s.select("motivo_chave", "nome_motivo", "categoria"), "motivo_chave", "left").drop("motivo_chave")
    .join(atendentes_s, "id_atendente", "left")
    .join(clientes_s, "id_cliente", "left")
    .withColumn("dq_cliente_sem_correspondencia", F.col("_cliente_encontrado").isNull())
    .drop("id_cliente", "_cliente_encontrado")
    .join(custo_por_chamado, "id_chamado", "left")
    .join(pesquisa_s.select("id_chamado", "nota", "nota_ajustada"), "id_chamado", "left")  # nota ja vem nula quando invalida (secao 6) - nao filtrar aqui, senao nota_ajustada perde as linhas resgatadas
    .join(hora_s, "id_chamado", "left")
)

(
    base_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_BASE)
)

print(f"{DESTINO_BASE}: {spark.table(DESTINO_BASE).count():,} linhas (esperado: 131.838)")

dados_vcredit.default.base_silver: 131,838 linhas (esperado: 131.838)


### Conferência

In [0]:
base = spark.table(DESTINO_BASE)
total = base.count()

print(f"com id_canal (join de canal):     {base.where('id_canal is not null').count():,} / {total:,}")
print(f"com nome_motivo (join de motivo): {base.where('nome_motivo is not null').count():,} / {total:,}")
print(f"com nivel_atendente:              {base.where('nivel_atendente is not null').count():,} / {total:,}")
print(f"com regiao (cliente):             {base.where('regiao is not null').count():,} / {total:,}")
print(f"com custo:                        {base.where('custo is not null').count():,} / {total:,} (esperado: 129.744 - so > 0)")
print(f"com nota:                         {base.where('nota is not null').count():,} / {total:,}")
print(f"com nota_ajustada:                {base.where('nota_ajustada is not null').count():,} / {total:,} (esperado: 74.592)")
print(f"com tempo_espera_seg:             {base.where('tempo_espera_seg is not null').count():,} / {total:,}")

sem_motivo = base.where("nome_motivo is null").count()
print(f"chamados sem motivo correspondente: {sem_motivo:,} (esperado: 0)")

# diferente de canal/motivo, aqui NAO esperamos 0 - e um achado real de qualidade de dado:
# ha id_cliente em chamados que nao existe em clientes.csv (nao e bug da juncao).
sem_cliente = base.where("dq_cliente_sem_correspondencia").count()
print(f"chamados sem cliente correspondente (dq_cliente_sem_correspondencia): {sem_cliente:,} ({sem_cliente/total:.1%} do total)")

com id_canal (join de canal):     131,838 / 131,838
com nome_motivo (join de motivo): 131,838 / 131,838
com nivel_atendente:              131,074 / 131,838
com regiao (cliente):             130,508 / 131,838
com custo:                        130,288 / 131,838 (esperado: 130.288)
com nota:                         73,383 / 131,838
com tempo_espera_seg:             128,424 / 131,838
chamados sem motivo correspondente: 0 (esperado: 0)
chamados sem cliente correspondente (dq_cliente_sem_correspondencia): 1,330 (1.0% do total)


## Checklist antes de aplicar uma mudança

- [ ] Contagens de cada seção batem com o esperado (impressas em cada célula) — `chamados_silver`
      com 2.636 duplicatas removidas, `base_silver` com 131.838 linhas.
- [ ] Nenhuma célula de output mostra linha de `clientes` (CPF/nome/e-mail) — nem na seção 2, nem na 8.
- [ ] 100% de correspondência em `id_canal` e `nome_motivo` na seção 8 (nenhum chamado sem dimensão).
- [ ] Rodamos duas vezes e o resultado foi o mesmo.